In [ ]:
#The comment under is a the command to install FluidSynth and midi2audio
#!apt-get install -y fluidsynth fluid-soundfont-gm -qq &&
#pip install midi2audio -q

import music21
from midi2audio import FluidSynth
from IPython.display import Audio, display, HTML

SF2 = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
#figured-bass suffix for each (inversion, has7) pair
FIGURES = {(0,0):"", (1,0):"6", (2,0):"64",
           (0,1):"7", (1,1):"65", (2,1):"43", (3,1):"42"}

degree_names    = {v: k for k, v in chord_map.items()}
duration_values = {v: k for k, v in dur_map.items()}


def render_audio(degree_ids, inversions, sevenths, duration_ids,
                 tonic="C", filename="out"):
    """Turn model codes into a wav file. Returns the path."""
    score = music21.stream.Stream()
    for deg, inv, has7, dur in zip(degree_ids, inversions,
                                   sevenths, duration_ids):
        figure = degree_names[deg] + FIGURES.get((int(inv), int(has7)), "")
        chord = music21.roman.RomanNumeral(figure, tonic)
        chord.quarterLength = max(float(duration_values[dur]), 0.25)
        score.append(chord)
    score.write("midi", fp=f"{filename}.mid")
    FluidSynth(SF2).midi_to_audio(f"{filename}.mid", f"{filename}.wav")
    return f"{filename}.wav"


#find where each chorale starts inside the test rows, so an excerpt
#never crosses from one chorale into the next
chorale_names = cgroup.index.tolist()
all_lengths   = cgroup["roman_scale_degree"].apply(len).tolist()

test_names   = [chorale_names[i] for i in location[test_slice]]
test_lengths = [all_lengths[i]   for i in location[test_slice]]

blocks, row = [], 0
for name, length in zip(test_names, test_lengths):
    n_windows = length - wsize
    if n_windows > 0:
        blocks.append((name, row, n_windows))
        row += n_windows


In [ ]:
#Converting a chorale and it's predicted version to .wav
chorale, start, n_windows = blocks[0]      # change the index for a different chorale
excerpt = slice(start, start + n_windows)

predictions = model.predict(X_test, verbose=0)

true_degrees = Y_test[0][excerpt]
true_inv     = Y_test[2][excerpt]
true_has7    = Y_test[3][excerpt]
true_dur     = Y_test[4][excerpt]

pred_degrees = predictions[0][excerpt].argmax(1)
pred_inv     = predictions[2][excerpt].argmax(1)
pred_has7    = predictions[3][excerpt].argmax(1)

#the first 8 wsize chords are the learning window: Bach's part and after that
#the prediction. Around 30 seconds of music will be produced.
seed_deg = X_test[0][start][:wsize]
seed_inv = X_test[2][start][:wsize]
seed_h7  = X_test[3][start][:wsize]
seed_dur = X_test[4][start][:wsize]

#Metrics to see the final percentage of the correctly predicted chords.
full_true_deg = np.concatenate([seed_deg, true_degrees])
full_true_inv = np.concatenate([seed_inv, true_inv])
full_true_h7  = np.concatenate([seed_h7,  true_has7])
full_dur      = np.concatenate([seed_dur, true_dur])

full_pred_deg = np.concatenate([seed_deg, pred_degrees])
full_pred_inv = np.concatenate([seed_inv, pred_inv])
full_pred_h7  = np.concatenate([seed_h7,  pred_has7])

match = (pred_degrees == true_degrees).mean()

display(HTML(f"<h3>{chorale}</h3>"
             f"<p>{len(full_dur)} chords &mdash; {match:.0%} of predicted "
             f"degrees match (first {wsize} are the shared opening)</p>"))
print("Bach:")
display(Audio(render_audio(full_true_deg, full_true_inv, full_true_h7,
                           full_dur, filename="bach")))
print("Model:")
display(Audio(render_audio(full_pred_deg, full_pred_inv, full_pred_h7,
                           full_dur, filename="model")))

In [ ]:
#Name of the chorale used to create the .wav file
names = cgroup.index.tolist()
test_names = [names[i] for i in location[test_slice]]
print(test_names[:5])
print("lengths:", [len(roman[i]) for i in range(3)])